# Task 1f — Numerical Validation

**Goal.** Validate the closed-form expression for $ec{P}(t)$ from part (d) and the asymptotic limit from part (e).

**Strategy.** Simulate the recursion $ec{P}(t+1) = \mathbb{M}\,ec{P}(t)$ directly, then compute $ec{P}(t)$ independently from the eigenmode decomposition (parts 1b–1c), and assert the two agree to machine precision. The asymptotic check then confirms $ec{P}(T) 	o ec{
u}_1 = (0,1)^	op$.


In [ ]:
import numpy as np

# --- Parameters ---
eps = 0.1                                 # conversion rate per generation
p0  = np.array([0.5, 0.5])                # initial state (p2 = 1 - p1)
T   = 100                                 # generations to simulate

# Markov matrix; lower-triangular, column-stochastic.
M = np.array([[1 - eps, 0.0],
              [eps,     1.0]])

# --- Stage 1: direct simulation ---
# Ground truth: iterate the recursion with no analytical input.
P_num = np.zeros((T + 1, 2))
P_num[0] = p0
for t_step in range(T):
    P_num[t_step + 1] = M @ P_num[t_step]

# --- Stage 2: closed form via eigenmode decomposition ---
# Eigenvectors from 1b (lambda_1 = 1, lambda_2 = 1 - eps).
v1 = np.array([0,  1])                    # persistent mode
v2 = np.array([1, -1])                    # decaying   mode

# Solve P(0) = c1*v1 + c2*v2 for the coefficients in the eigenbasis.
c1, c2 = np.linalg.solve(np.column_stack([v1, v2]), p0)
print(f"c1 = {c1},  c2 = {c2}")           # expect c1 = 1, c2 = 0.5

# Evaluate the closed form for all t at once via broadcasting.
t = np.arange(T + 1)
P_closed = c1 * v1 + c2 * ((1 - eps) ** t)[:, None] * v2

# --- Stage 3: validation ---
# Both arrays must agree to machine precision; the assertion catches any
# future edit that breaks the closed form.
max_err = np.max(np.abs(P_num - P_closed))
print(f"max |numeric - closed form| = {max_err:.2e}")
assert max_err < 1e-10, "closed form disagrees with simulation -- check 1d"

# Asymptotic check (validates 1e): P(T) sits near [0, 1] = nu_1.
print(f"P(T={T}) = {P_num[-1]}   (should approach [0, 1])")


## Visualisation

Two-panel figure for the report:

- **Panel A (linear scale).** Trajectories $p_1(t), p_2(t)$ converging to $(0, 1)$. Visual confirmation of part (e); dotted horizontal lines mark the asymptotes.
- **Panel B (log scale).** Numeric $p_1(t)$ overlaid on the closed form $rac{1}{2}(1-\epsilon)^t$. The straight line on semi-log $y$ axes proves the decay is geometric with rate $1-\epsilon$ (validates the rate); numeric dots sitting on top of the line validate the closed form from (d).


In [ ]:
import matplotlib.pyplot as plt

# Match the report's serif typesetting.
plt.rcParams.update({"font.family": "serif", "font.size": 11})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel A: linear-scale trajectories ---
# Dotted horizontal lines mark the asymptotic values, making the limit
# in 1e visible at a glance.
ax = axes[0]
ax.plot(t, P_num[:, 0], "o-", color="tab:blue",   ms=3, lw=1,
        label=r"$p_1(t)$ (type 1)")
ax.plot(t, P_num[:, 1], "o-", color="tab:orange", ms=3, lw=1,
        label=r"$p_2(t)$ (type 2)")
ax.axhline(0, color="tab:blue",   ls=":", lw=1, alpha=0.6)
ax.axhline(1, color="tab:orange", ls=":", lw=1, alpha=0.6)
ax.set_xlabel(r"generation $t$")
ax.set_ylabel("fractional population")
ax.set_title(rf"Population trajectories ($\epsilon = {eps}$)")
ax.set_ylim(-0.02, 1.05)
ax.legend(loc="center right")
ax.grid(alpha=0.3)

# --- Panel B: log-scale; numeric vs closed form ---
# A straight line on semilog-y proves the decay is geometric with rate
# (1 - eps); numeric dots sitting on the line validate the closed form.
theory_p1 = 0.5 * (1 - eps) ** t          # (1/2)(1-eps)^t

ax = axes[1]
ax.semilogy(t, theory_p1,   "-",  color="black",    lw=1.2,
            label=r"theory: $\frac{1}{2}(1-\epsilon)^t$")
ax.semilogy(t, P_num[:, 0], "o",  color="tab:blue", ms=4,
            label=r"numeric $p_1(t)$")
ax.set_xlabel(r"generation $t$")
ax.set_ylabel(r"$p_1(t)$  (log scale)")
ax.set_title("Geometric decay of type-1: numeric vs closed form")
ax.legend()
ax.grid(which="both", alpha=0.3)

fig.tight_layout()
fig.savefig("task1f_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()
